# MVP Pipeline — Data Streams → Desired Position

This notebook prototypes the core server-side pipeline (MVP steps 4–6) that converts
standardised data streams into a single **desired position** per risk dimension.

---

## Glossary

| Term | Meaning |
|------|---------|
| **Risk dimension** | A unique grouping for which we compute a desired position. Defined by `risk_dimension_cols` (e.g. `["symbol", "expiry"]`). |
| **Space** | A temporal bucket within a risk dimension. Spaces are defined by their start time — "shifting" spaces always start at *now*; "static" spaces start at a fixed timestamp (e.g. an event). |
| **Stream** | A named data feed (e.g. realised vol, mean IV, event moves). Each stream produces one or more **blocks**. |
| **Block** | One row of a stream's latest snapshot, uniquely keyed by `key_cols`. A block maps to exactly one space and carries a fair value through time. `key_cols` is the full index of the snapshot (including risk dimension cols plus any extra keys like `event_id`). |
| **Target-space conversion** | Transforms `raw_value` from the stream's native units into the risk dimension's units (e.g. vol → variance via squaring). |
| **Fair value** | Our estimate of the true value in a space, distributed through time according to block config (decay, annualisation). |
| **Market-implied value** | The market's pricing of a space, distributed through time using the same block shape. |
| **Variance** | Uncertainty of our fair value estimate. Proportional to `abs(fair) * var_fair_ratio`. |
| **Edge** | `fair − market_fair`, aggregated per space then summed across spaces within a risk dimension. |
| **Desired position** | `edge / variance * bankroll`, optionally smoothed. One value per risk dimension per timestamp. |

---

## How to use this notebook

1. **Edit scenario config** (Cell 4) to change `now`, `bankroll`, `risk_dimension_cols`, etc.
2. **Edit stream configs** (Cell 5) to add/remove/modify streams and their mock data. Streams carry their own `symbol`, `expiry`, etc.
3. **Run All** — the pipeline is fully top-to-bottom; no hidden state or cell-order dependencies.
4. **Inspect intermediates** — `run_pipeline()` returns a dict with every stage's DataFrame.
5. **Sensecheck** — the final cells plot fair value, variance, edge, and desired position.

In [39]:
from __future__ import annotations

import datetime as dt
from dataclasses import dataclass, field
from typing import Literal

import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

## Configuration

In [40]:
SECONDS_PER_YEAR = 365.25 * 24 * 60 * 60


@dataclass(frozen=True)
class BlockConfig:
    """Immutable configuration for how a single block distributes value through time.

    These parameters are set per-stream and inherited by every block that stream produces.
    The only per-block override is ``start_timestamp`` (read from the snapshot row).
    """
    annualized: bool = True
    size_type: Literal["fixed", "relative"] = "fixed"
    aggregation_logic: Literal["average", "offset"] = "average"
    temporal_position: Literal["static", "shifting"] = "shifting"
    decay_end_size_mult: float = 1.0
    decay_rate_prop_per_min: float = 0.0
    decay_profile: Literal["linear"] = "linear"
    var_fair_ratio: float = 1.0

    def __post_init__(self):
        assert self.size_type in ("fixed", "relative")
        assert self.aggregation_logic in ("average", "offset")
        assert self.temporal_position in ("static", "shifting")
        assert self.decay_profile in ("linear",)
        assert self.decay_end_size_mult >= 0
        assert self.decay_rate_prop_per_min >= 0
        if self.size_type == "relative":
            assert self.annualized, "must be annualized for relative size"
        if self.decay_end_size_mult != 0 and not self.annualized:
            raise ValueError("decay_end_size_mult is only applicable for annualized streams")


@dataclass(frozen=True)
class StreamConfig:
    """Immutable specification of a data stream.

    ``snapshot`` is the raw DataFrame the stream provides.  It must contain all
    ``key_cols`` plus ``timestamp`` and ``raw_value``.

    ``key_cols`` is the full index of the snapshot — used to deduplicate to the
    latest row per key.  It must be a superset of the pipeline's
    ``risk_dimension_cols`` (e.g. ``["symbol", "expiry"]``), plus any extra keys
    specific to the stream (e.g. ``"event_id"``).
    """
    stream_name: str
    snapshot: pl.DataFrame
    key_cols: list[str]
    scale: float = 1.0
    offset: float = 0.0
    exponent: float = 1.0
    block: BlockConfig = field(default_factory=BlockConfig)

In [41]:
# ── Scenario parameters (edit this cell to change the scenario) ──────────
now = dt.datetime(2026, 1, 1)
bankroll = 100_000
smoothing_hl_secs = 60 * 15   # EWM half-life for position smoothing
time_grid_interval = "1m"     # resolution of the future time grid

# Columns that define a unique risk dimension.
# Each unique combination gets its own desired position curve.
# These must be present in every stream's key_cols.
risk_dimension_cols: list[str] = ["symbol", "expiry"]

In [42]:
# ── Stream definitions (edit this cell to add / change streams) ──────────
# symbol and expiry live inside each stream's snapshot — not as globals.

_symbol = "BTC"
_expiry = dt.datetime(2026, 1, 2)

rv_stream = StreamConfig(
    stream_name="rv",
    snapshot=pl.DataFrame({
        "timestamp": [now],
        "symbol": [_symbol],
        "expiry": [_expiry],
        "raw_value": [0.45],
    }),
    key_cols=["symbol", "expiry"],
    scale=1.0, offset=0.0, exponent=2,
    block=BlockConfig(annualized=True),
)

mean_iv_stream = StreamConfig(
    stream_name="mean_iv",
    snapshot=pl.DataFrame({
        "timestamp": [now],
        "symbol": [_symbol],
        "expiry": [_expiry],
        "raw_value": [0.50],
    }),
    key_cols=["symbol", "expiry"],
    scale=1.0, offset=0.0, exponent=2,
    block=BlockConfig(
        annualized=True,
        aggregation_logic="offset",
        size_type="relative",
        decay_end_size_mult=0.0,
        decay_rate_prop_per_min=0.01,
        var_fair_ratio=2.0,
    ),
)

num_events = 5
event_starts = [now + dt.timedelta(hours=i * 4) for i in range(num_events)]
events_stream = StreamConfig(
    stream_name="events",
    snapshot=pl.DataFrame({
        "timestamp": [now] * num_events,
        "symbol": [_symbol] * num_events,
        "expiry": [_expiry] * num_events,
        "event_id": [f"event_{i}" for i in range(num_events)],
        "raw_value": [2.5, 3.1, 1.8, 4.0, 2.0],
        "start_timestamp": event_starts,
    }),
    key_cols=["symbol", "expiry", "event_id"],
    scale=1 / 100, offset=0.0, exponent=2,
    block=BlockConfig(
        annualized=False,
        aggregation_logic="offset",
        temporal_position="static",
        decay_end_size_mult=0.0,
        decay_rate_prop_per_min=0.01,
        var_fair_ratio=3.0,
    ),
)

# Mock market-implied pricing per space (keyed by space_id).
# In production this comes from the data feed; here we use deterministic values.
mock_market_pricing: dict[str, float] = {
    "shifting": 0.55,
    "static_20260101_000000": 0.30,
    "static_20260101_040000": 0.25,
    "static_20260101_080000": 0.40,
    "static_20260101_120000": 0.35,
    "static_20260101_160000": 0.20,
}

STREAMS: list[StreamConfig] = [rv_stream, mean_iv_stream, events_stream]

## Helper Functions

In [43]:
def annualize(value: float, seconds: float) -> float:
    """Convert a total value over ``seconds`` to an annualised rate."""
    return value * SECONDS_PER_YEAR / seconds


def deannualize(value: float, seconds: float) -> float:
    """Convert an annualised rate to a total value over ``seconds``."""
    return value / SECONDS_PER_YEAR * seconds


def raw_to_target_expr(col: str, scale: float, offset: float, exponent: float) -> pl.Expr:
    """Native Polars expression for target-space conversion (no map_elements)."""
    return (scale * pl.col(col) + offset).pow(exponent)

## Pipeline Steps

In [44]:
def build_blocks_df(
    streams: list[StreamConfig],
    risk_dimension_cols: list[str],
) -> pl.DataFrame:
    """Flatten stream configs into one row per block with all needed columns.

    Validates that every stream's ``key_cols`` is a superset of ``risk_dimension_cols``.

    Returns a DataFrame with columns including all risk_dimension_cols,
    block_name, stream_name, target_value, space_id, and flat BlockConfig fields.
    """
    rows: list[dict] = []
    for sc in streams:
        missing = set(risk_dimension_cols) - set(sc.key_cols)
        if missing:
            raise ValueError(
                f"Stream '{sc.stream_name}' key_cols {sc.key_cols} "
                f"missing risk_dimension_cols: {missing}"
            )

        extra_keys = [k for k in sc.key_cols if k not in risk_dimension_cols]
        snap = sc.snapshot.sort("timestamp").group_by(sc.key_cols).agg(pl.all().last())

        # Ensure start_timestamp column exists
        if "start_timestamp" not in snap.columns:
            snap = snap.with_columns(pl.lit(None).cast(pl.Datetime("us")).alias("start_timestamp"))

        # Native Polars target-space conversion
        snap = snap.with_columns(
            raw_to_target_expr("raw_value", sc.scale, sc.offset, sc.exponent).alias("target_value"),
        )

        for row in snap.iter_rows(named=True):
            block_name = "_".join([sc.stream_name] + [str(row[k]) for k in extra_keys])

            # Assign space_id
            if sc.block.temporal_position == "shifting":
                space_id = "shifting"
            else:
                st = row["start_timestamp"]
                if st is None:
                    raise ValueError(f"start_timestamp required for static block {block_name}")
                space_id = f"static_{st.strftime('%Y%m%d_%H%M%S')}"

            entry = {
                "block_name": block_name,
                "stream_name": sc.stream_name,
                "target_value": row["target_value"],
                "raw_value": row["raw_value"],
                "start_timestamp": row.get("start_timestamp"),
                "space_id": space_id,
                # Block config fields (flat)
                "annualized": sc.block.annualized,
                "size_type": sc.block.size_type,
                "aggregation_logic": sc.block.aggregation_logic,
                "temporal_position": sc.block.temporal_position,
                "decay_end_size_mult": sc.block.decay_end_size_mult,
                "decay_rate_prop_per_min": sc.block.decay_rate_prop_per_min,
                "var_fair_ratio": sc.block.var_fair_ratio,
                "scale": sc.scale,
                "offset": sc.offset,
                "exponent": sc.exponent,
            }
            # Carry through all risk dimension cols from the row
            for rdc in risk_dimension_cols:
                entry[rdc] = row[rdc]

            rows.append(entry)

    return pl.DataFrame(rows)

In [45]:
def attach_market_values(
    blocks_df: pl.DataFrame,
    market_pricing: dict[str, float],
) -> pl.DataFrame:
    """Join market-implied values and convert to target space.

    Validation checks:
    - Every space_id in blocks_df has a market price.
    - No null target_values.
    """
    missing = set(blocks_df["space_id"].unique().to_list()) - set(market_pricing.keys())
    if missing:
        raise ValueError(f"Missing market pricing for spaces: {missing}")

    market_df = pl.DataFrame({
        "space_id": list(market_pricing.keys()),
        "market_value": list(market_pricing.values()),
    })

    out = blocks_df.join(market_df, on="space_id", how="left")
    out = out.with_columns(
        (pl.col("scale") * pl.col("market_value") + pl.col("offset"))
        .pow(pl.col("exponent"))
        .alias("target_market_value"),
    )

    # Validation
    null_count = out.filter(pl.col("target_value").is_null()).height
    if null_count > 0:
        raise ValueError(f"{null_count} blocks have null target_value")

    return out

In [46]:
def build_time_grid(
    blocks_df: pl.DataFrame,
    risk_dimension_cols: list[str],
    now: dt.datetime,
    interval: str = "1m",
) -> pl.DataFrame:
    """Create a time grid per unique risk dimension.

    Discovers ``expiry`` from the blocks_df (must be in ``risk_dimension_cols`` or
    as a column).  One grid is built from ``now`` to each risk dimension's expiry.

    Returns a single DataFrame with risk_dimension_cols + timestamp + dtte.
    """
    if "expiry" not in blocks_df.columns:
        raise ValueError("blocks_df must contain an 'expiry' column to build time grids")

    unique_dims = blocks_df.select(risk_dimension_cols).unique()
    parts: list[pl.DataFrame] = []

    for row in unique_dims.iter_rows(named=True):
        expiry = row["expiry"]
        timestamps = pl.datetime_range(start=now, end=expiry, interval=interval, eager=True)
        grid = pl.DataFrame({"timestamp": timestamps})

        # Add risk dimension columns as literals
        for rdc in risk_dimension_cols:
            grid = grid.with_columns(pl.lit(row[rdc]).alias(rdc))

        grid = grid.with_columns(
            dtte=-pl.col("timestamp").diff(-1).dt.total_seconds() / SECONDS_PER_YEAR,
        )
        parts.append(grid)

    return pl.concat(parts)

In [47]:
def _get_end_timestamp(
    start_ts: dt.datetime,
    expiry: dt.datetime,
    decay_end_size_mult: float,
    decay_rate_prop_per_min: float,
) -> dt.datetime:
    if decay_end_size_mult == 1 or decay_rate_prop_per_min == 0:
        return expiry
    return start_ts + dt.timedelta(minutes=1 / decay_rate_prop_per_min)


def _get_total_value(
    stream_value: float,
    market_value: float,
    start_ts: dt.datetime,
    end_ts: dt.datetime,
    is_annualized: bool,
    size_type: str,
) -> float:
    if is_annualized:
        ann_val = stream_value if size_type == "fixed" else stream_value - market_value
        return deannualize(ann_val, (end_ts - start_ts).total_seconds())
    return stream_value


def _get_start_annualized_value(
    total_value: float,
    expiry: dt.datetime,
    start_ts: dt.datetime,
    end_ts: dt.datetime,
    end_annualized_value: float,
    is_annualized: bool,
) -> float:
    start_to_expiry_secs = (expiry - start_ts).total_seconds()
    start_to_end_secs = (end_ts - start_ts).total_seconds()
    ann_val = annualize(total_value, start_to_end_secs)

    if is_annualized:
        start_to_end_secs = min(start_to_expiry_secs, start_to_end_secs)
        # v = p*(s+e)/2 + (1-p)*e  =>  s = (2/p)*(v - (1-p)*e) - e
        p = start_to_end_secs / start_to_expiry_secs
        return (2 / p) * (ann_val - (1 - p) * end_annualized_value) - end_annualized_value
    return ann_val


def compute_block_fair_values(
    blocks_df: pl.DataFrame,
    time_grid: pl.DataFrame,
    risk_dimension_cols: list[str],
    now: dt.datetime,
) -> pl.DataFrame:
    """Compute fair_annualized and fair for every (timestamp, block_name).

    Reads ``expiry`` from each block's row in blocks_df.
    Joins each block to its risk dimension's time grid.

    Returns a **long-format** DataFrame with risk_dimension_cols +
        timestamp | block_name | stream_name | space_id | aggregation_logic |
        fair_annualized | fair | market_fair | var_fair_ratio
    """
    parts: list[pl.DataFrame] = []

    for row in blocks_df.iter_rows(named=True):
        block_name = row["block_name"]
        is_ann = row["annualized"]
        expiry = row["expiry"]
        start_ts = now if row["temporal_position"] == "shifting" else row["start_timestamp"]
        end_ts = _get_end_timestamp(start_ts, expiry, row["decay_end_size_mult"], row["decay_rate_prop_per_min"])

        target_val = row["target_value"]
        target_mkt = row["target_market_value"]

        total_val = _get_total_value(target_val, target_mkt, start_ts, end_ts, is_ann, row["size_type"])
        dur_secs = (end_ts - start_ts).total_seconds()
        end_ann = annualize(total_val, dur_secs) * row["decay_end_size_mult"]
        start_ann = _get_start_annualized_value(total_val, expiry, start_ts, end_ts, end_ann, is_ann)

        # Market fair uses the same block shape but with target_market_value as input
        mkt_total = _get_total_value(target_mkt, target_mkt, start_ts, end_ts, is_ann, row["size_type"])
        mkt_end_ann = annualize(mkt_total, dur_secs) * row["decay_end_size_mult"]
        mkt_start_ann = _get_start_annualized_value(mkt_total, expiry, start_ts, end_ts, mkt_end_ann, is_ann)

        # Filter the time grid to this block's risk dimension
        grid_filter = time_grid
        for rdc in risk_dimension_cols:
            grid_filter = grid_filter.filter(pl.col(rdc) == row[rdc])

        block_df = grid_filter.select(risk_dimension_cols + ["timestamp", "dtte"]).with_columns(
            # Fair annualized
            (
                pl.when(pl.col("timestamp") < start_ts).then(0.0)
                .when(pl.col("timestamp") > end_ts).then(end_ann)
                .when(pl.lit(is_ann)).then(
                    start_ann + (end_ann - start_ann)
                    * (pl.col("timestamp") - start_ts) / (end_ts - start_ts)
                )
                .otherwise(start_ann)
            ).alias("fair_annualized"),
            # Market fair annualized
            (
                pl.when(pl.col("timestamp") < start_ts).then(0.0)
                .when(pl.col("timestamp") > end_ts).then(mkt_end_ann)
                .when(pl.lit(is_ann)).then(
                    mkt_start_ann + (mkt_end_ann - mkt_start_ann)
                    * (pl.col("timestamp") - start_ts) / (end_ts - start_ts)
                )
                .otherwise(mkt_start_ann)
            ).alias("market_fair_annualized"),
            # Static metadata
            pl.lit(block_name).alias("block_name"),
            pl.lit(row["stream_name"]).alias("stream_name"),
            pl.lit(row["space_id"]).alias("space_id"),
            pl.lit(row["aggregation_logic"]).alias("aggregation_logic"),
            pl.lit(row["var_fair_ratio"]).alias("var_fair_ratio"),
        ).with_columns(
            (pl.col("fair_annualized") * pl.col("dtte")).alias("fair"),
            (pl.col("market_fair_annualized") * pl.col("dtte")).alias("market_fair"),
        )

        parts.append(block_df)

    return pl.concat(parts)

In [48]:
def compute_block_variances(block_fair_df: pl.DataFrame) -> pl.DataFrame:
    """Add a ``var`` column: variance = abs(fair) * var_fair_ratio."""
    return block_fair_df.with_columns(
        (pl.col("fair").abs() * pl.col("var_fair_ratio")).alias("var"),
    )

In [49]:
def aggregate_by_space(
    block_df: pl.DataFrame,
    risk_dimension_cols: list[str],
) -> pl.DataFrame:
    """Aggregate block-level fair / market_fair / var to space-level, then sum across spaces.

    Groups by ``risk_dimension_cols`` so each risk dimension gets its own aggregation.

    Aggregation rules per space:
      - 'average' blocks: mean of fair values
      - 'offset' blocks:  sum of fair values
      - space_fair = average_fair + offset_fair
      - space_var  = sum of all block variances (no averaging)
      - edge = space_fair - space_market_fair  (per space, then summed)

    Returns a DataFrame with risk_dimension_cols + timestamp + total_fair + total_market_fair + edge + var.
    """
    group_keys = risk_dimension_cols + ["timestamp", "space_id"]

    avg_df = block_df.filter(pl.col("aggregation_logic") == "average")
    off_df = block_df.filter(pl.col("aggregation_logic") == "offset")

    if avg_df.height > 0:
        avg_agg = avg_df.group_by(group_keys).agg(
            pl.col("fair").mean().alias("avg_fair"),
            pl.col("market_fair").mean().alias("avg_market_fair"),
        )
    else:
        schema = {c: block_df.schema[c] for c in group_keys}
        schema.update({"avg_fair": pl.Float64, "avg_market_fair": pl.Float64})
        avg_agg = pl.DataFrame(schema=schema)

    if off_df.height > 0:
        off_agg = off_df.group_by(group_keys).agg(
            pl.col("fair").sum().alias("off_fair"),
            pl.col("market_fair").sum().alias("off_market_fair"),
        )
    else:
        schema = {c: block_df.schema[c] for c in group_keys}
        schema.update({"off_fair": pl.Float64, "off_market_fair": pl.Float64})
        off_agg = pl.DataFrame(schema=schema)

    var_agg = block_df.group_by(group_keys).agg(
        pl.col("var").sum().alias("space_var"),
    )

    space_df = var_agg.join(avg_agg, on=group_keys, how="left")
    space_df = space_df.join(off_agg, on=group_keys, how="left")
    space_df = space_df.with_columns(
        pl.col("avg_fair").fill_null(0.0),
        pl.col("avg_market_fair").fill_null(0.0),
        pl.col("off_fair").fill_null(0.0),
        pl.col("off_market_fair").fill_null(0.0),
    ).with_columns(
        (pl.col("avg_fair") + pl.col("off_fair")).alias("space_fair"),
        (pl.col("avg_market_fair") + pl.col("off_market_fair")).alias("space_market_fair"),
    ).with_columns(
        (pl.col("space_fair") - pl.col("space_market_fair")).alias("space_edge"),
    )

    rd_ts_keys = risk_dimension_cols + ["timestamp"]
    total = space_df.group_by(rd_ts_keys).agg(
        pl.col("space_fair").sum().alias("total_fair"),
        pl.col("space_market_fair").sum().alias("total_market_fair"),
        pl.col("space_edge").sum().alias("edge"),
        pl.col("space_var").sum().alias("var"),
    ).sort(rd_ts_keys)

    return total

In [50]:
def compute_desired_position(
    agg_df: pl.DataFrame,
    risk_dimension_cols: list[str],
    bankroll: float,
    smoothing_hl_secs: int,
) -> pl.DataFrame:
    """Derive raw and smoothed desired position from edge and variance.

    Smoothing is applied independently per risk dimension.
    Guards against division by near-zero variance.
    """
    VAR_FLOOR = 1e-18

    out = agg_df.with_columns(
        pl.when(pl.col("var").abs() < VAR_FLOOR)
        .then(0.0)
        .otherwise(pl.col("edge") * bankroll / pl.col("var"))
        .alias("raw_desired_position"),
    )

    # Smooth per risk dimension
    smoothed_parts: list[pl.DataFrame] = []
    for _, group in out.group_by(risk_dimension_cols):
        group = group.sort("timestamp").with_columns(
            pl.col("raw_desired_position")
            .reverse()
            .ewm_mean_by("timestamp", half_life=f"{smoothing_hl_secs}s")
            .reverse()
            .alias("smoothed_desired_position"),
        )
        smoothed_parts.append(group)

    return pl.concat(smoothed_parts).sort(risk_dimension_cols + ["timestamp"])

In [51]:
def run_pipeline(
    streams: list[StreamConfig],
    market_pricing: dict[str, float],
    risk_dimension_cols: list[str],
    now: dt.datetime,
    bankroll: float,
    smoothing_hl_secs: int,
    time_grid_interval: str = "1m",
) -> dict[str, pl.DataFrame]:
    """Execute the full pipeline and return every intermediate as a dict.

    Keys:
        blocks_df        — one row per block (flat config + target values)
        time_grid        — timestamp grid per risk dimension
        block_fair_df    — long-format fair / market_fair per (timestamp, block)
        block_var_df     — above + variance column
        space_agg_df     — one row per (risk_dimension, timestamp), aggregated across spaces
        desired_pos_df   — above + raw & smoothed desired position
    """
    blocks_df = build_blocks_df(streams, risk_dimension_cols)
    blocks_df = attach_market_values(blocks_df, market_pricing)
    time_grid = build_time_grid(blocks_df, risk_dimension_cols, now, interval=time_grid_interval)
    block_fair_df = compute_block_fair_values(blocks_df, time_grid, risk_dimension_cols, now)
    block_var_df = compute_block_variances(block_fair_df)
    space_agg_df = aggregate_by_space(block_var_df, risk_dimension_cols)
    desired_pos_df = compute_desired_position(space_agg_df, risk_dimension_cols, bankroll, smoothing_hl_secs)

    return {
        "blocks_df": blocks_df,
        "time_grid": time_grid,
        "block_fair_df": block_fair_df,
        "block_var_df": block_var_df,
        "space_agg_df": space_agg_df,
        "desired_pos_df": desired_pos_df,
    }

## Run Pipeline

In [52]:
results = run_pipeline(
    streams=STREAMS,
    market_pricing=mock_market_pricing,
    risk_dimension_cols=risk_dimension_cols,
    now=now,
    bankroll=bankroll,
    smoothing_hl_secs=smoothing_hl_secs,
    time_grid_interval=time_grid_interval,
)

print("=== Block Summary ===")
display(results["blocks_df"].select(
    risk_dimension_cols + [
        "block_name", "stream_name", "space_id", "aggregation_logic",
        "target_value", "target_market_value", "var_fair_ratio",
    ]
))

print("\n=== Time Grid (head) ===")
display(results["time_grid"].head())

print("\n=== Block Fair Values — long format (head) ===")
display(results["block_var_df"].head(10))

print("\n=== Desired Position (head) ===")
display(results["desired_pos_df"].head())

=== Block Summary ===


symbol,expiry,block_name,stream_name,space_id,aggregation_logic,target_value,target_market_value,var_fair_ratio
str,datetime[μs],str,str,str,str,f64,f64,f64
"""BTC""",2026-01-02 00:00:00,"""rv""","""rv""","""shifting""","""average""",0.2025,0.3025,1.0
"""BTC""",2026-01-02 00:00:00,"""mean_iv""","""mean_iv""","""shifting""","""offset""",0.25,0.3025,2.0
"""BTC""",2026-01-02 00:00:00,"""events_event_2""","""events""","""static_20260101_080000""","""offset""",0.000324,0.000016,3.0
"""BTC""",2026-01-02 00:00:00,"""events_event_0""","""events""","""static_20260101_000000""","""offset""",0.000625,0.000009,3.0
"""BTC""",2026-01-02 00:00:00,"""events_event_4""","""events""","""static_20260101_160000""","""offset""",0.0004,0.000004,3.0
"""BTC""",2026-01-02 00:00:00,"""events_event_3""","""events""","""static_20260101_120000""","""offset""",0.0016,0.000012,3.0
"""BTC""",2026-01-02 00:00:00,"""events_event_1""","""events""","""static_20260101_040000""","""offset""",0.000961,0.000006,3.0



=== Time Grid (head) ===


timestamp,symbol,expiry,dtte
datetime[μs],str,datetime[μs],f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:01:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:02:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:03:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:04:00,"""BTC""",2026-01-02 00:00:00,0.000002



=== Block Fair Values — long format (head) ===


symbol,expiry,timestamp,dtte,fair_annualized,market_fair_annualized,block_name,stream_name,space_id,aggregation_logic,var_fair_ratio,fair,market_fair,var
str,datetime[μs],datetime[μs],f64,f64,f64,str,str,str,str,f64,f64,f64,f64
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:01:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:02:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:03:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:04:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:05:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:06:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:07:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:08:00,0.000002,0.2025,0.3025,"""rv""","""rv""","""shifting""","""average""",1.0,3.8501e-7,5.7514e-7,3.8501e-7



=== Desired Position (head) ===


symbol,expiry,timestamp,total_fair,total_market_fair,edge,var,raw_desired_position,smoothed_desired_position
str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.000004,6.6514e-7,0.000003,0.000025,12437.977575,15036.349197
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:01:00,0.000004,6.6514e-7,0.000003,0.000025,12582.572689,15159.236903
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:02:00,0.000004,6.6514e-7,0.000003,0.000025,12727.83907,15281.097977
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:03:00,0.000004,6.6514e-7,0.000003,0.000025,12873.781403,15401.852117
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:04:00,0.000004,6.6514e-7,0.000003,0.000025,13020.404418,15521.415004


In [53]:
# ── Validation summary ───────────────────────────────────────────────────
pos_df = results["desired_pos_df"]
block_var_df = results["block_var_df"]

checks = {
    "total timestamps": pos_df.height,
    "null edge rows": pos_df.filter(pl.col("edge").is_null()).height,
    "null var rows": pos_df.filter(pl.col("var").is_null()).height,
    "zero var rows": pos_df.filter(pl.col("var").abs() < 1e-18).height,
    "null fair blocks": block_var_df.filter(pl.col("fair").is_null()).height,
    "unique blocks": block_var_df["block_name"].n_unique(),
    "unique spaces": block_var_df["space_id"].n_unique(),
    "unique risk dimensions": pos_df.select(risk_dimension_cols).unique().height,
}
print("=== Validation Checks ===")
for k, v in checks.items():
    status = "OK" if ("null" in k or "zero" in k) and v == 0 else ("WARN" if ("null" in k or "zero" in k) and v > 0 else "INFO")
    print(f"  [{status}] {k}: {v}")

=== Validation Checks ===
  [INFO] total timestamps: 1441
  [OK] null edge rows: 0
  [OK] null var rows: 0
  [WARN] zero var rows: 1
  [WARN] null fair blocks: 7
  [INFO] unique blocks: 7
  [INFO] unique spaces: 6
  [INFO] unique risk dimensions: 1


## Sensechecking

In [54]:
# Graph 1: Fair Value by Space vs Market Implied
# (Plots the first risk dimension found; extend the loop for multi-dim)
block_var_df = results["block_var_df"]
plot_block = block_var_df.drop_nulls(subset=["fair"])

# Pick first risk dimension for plotting
first_rd = plot_block.select(risk_dimension_cols).unique().sort(risk_dimension_cols).row(0, named=True)
rd_label = " / ".join(f"{k}={v}" for k, v in first_rd.items())
rd_filter = plot_block
for k, v in first_rd.items():
    rd_filter = rd_filter.filter(pl.col(k) == v)

space_ids = sorted(
    rd_filter["space_id"].unique().to_list(),
    key=lambda s: (s != "shifting", s),
)

# Aggregate per (timestamp, space_id) for plotting
avg_part = (
    rd_filter.filter(pl.col("aggregation_logic") == "average")
    .group_by("timestamp", "space_id")
    .agg(pl.col("fair").mean().alias("avg_fair"), pl.col("market_fair").mean().alias("avg_mkt"))
)
off_part = (
    rd_filter.filter(pl.col("aggregation_logic") == "offset")
    .group_by("timestamp", "space_id")
    .agg(pl.col("fair").sum().alias("off_fair"), pl.col("market_fair").sum().alias("off_mkt"))
)

space_plot = (
    avg_part.join(off_part, on=["timestamp", "space_id"], how="outer_coalesce")
    .with_columns(
        pl.col("avg_fair").fill_null(0.0),
        pl.col("off_fair").fill_null(0.0),
        pl.col("avg_mkt").fill_null(0.0),
        pl.col("off_mkt").fill_null(0.0),
    )
    .with_columns(
        (pl.col("avg_fair") + pl.col("off_fair")).alias("total_fair"),
        (pl.col("avg_mkt") + pl.col("off_mkt")).alias("total_mkt"),
    )
    .sort("timestamp")
)

fig1 = make_subplots(
    rows=len(space_ids), cols=1, shared_xaxes=True,
    subplot_titles=space_ids, vertical_spacing=0.04,
)

for row_idx, sid in enumerate(space_ids, start=1):
    sp = space_plot.filter(pl.col("space_id") == sid)
    ts = sp["timestamp"].to_list()
    fig1.add_trace(go.Scatter(
        x=ts, y=sp["total_mkt"].to_list(), name="market fair",
        legendgroup="market fair", showlegend=row_idx == 1,
        line=dict(dash="dash", color="rgba(255,255,255,0.6)", width=2),
    ), row=row_idx, col=1)
    fig1.add_trace(go.Scatter(
        x=ts, y=sp["total_fair"].to_list(), name="total fair",
        legendgroup="total fair", showlegend=row_idx == 1,
        line=dict(width=2, color="rgba(0,204,150,1)"),
    ), row=row_idx, col=1)

fig1.update_layout(title=f"Fair Value by Space vs Market Fair — {rd_label}", template="plotly_dark", height=260 * len(space_ids))
fig1.update_xaxes(title_text="Time", row=len(space_ids), col=1)
fig1.update_yaxes(title_text="Fair x dtte")
fig1.show()

/var/folders/hq/l4cpmsnj2lq8xpc90xr2bt2w0000gn/T/ipykernel_80171/3816906707.py:31: DeprecationWarning: use of `how='outer_coalesce'` should be replaced with `how='full', coalesce=True`.
(Deprecated in version 0.20.29)
  avg_part.join(off_part, on=["timestamp", "space_id"], how="outer_coalesce")


In [55]:
# Graph 2: Stacked Variance Contribution by Block
fig2 = go.Figure()

for sid in space_ids:
    sp_blocks = rd_filter.filter(pl.col("space_id") == sid)
    for bn in sp_blocks["block_name"].unique().sort().to_list():
        bdata = sp_blocks.filter(pl.col("block_name") == bn).sort("timestamp")
        fig2.add_trace(go.Scatter(
            x=bdata["timestamp"].to_list(),
            y=bdata["var"].to_list(),
            name=f"{sid} | {bn}",
            stackgroup="one",
            legendgroup=sid,
        ))

fig2.update_layout(
    title=f"Variance Contribution by Block — {rd_label}",
    template="plotly_dark",
    xaxis_title="Time",
    yaxis_title="Variance",
)
fig2.show()

In [56]:
# Graph 3: Edge by Space
fig3 = make_subplots(
    rows=len(space_ids), cols=1, shared_xaxes=True,
    subplot_titles=space_ids, vertical_spacing=0.04,
)

for row_idx, sid in enumerate(space_ids, start=1):
    sp = space_plot.filter(pl.col("space_id") == sid)
    edge = (sp["total_fair"] - sp["total_mkt"]).to_list()
    fig3.add_trace(go.Scatter(
        x=sp["timestamp"].to_list(), y=edge,
        name=f"{sid} edge", showlegend=False,
    ), row=row_idx, col=1)

fig3.update_layout(title=f"Edge by Space — {rd_label}", template="plotly_dark", height=260 * len(space_ids))
fig3.update_xaxes(title_text="Time", row=len(space_ids), col=1)
fig3.update_yaxes(title_text="Fair - Market Fair")
fig3.show()

In [57]:
# Graph 4: Raw and Smoothed Desired Position
pos_df = results["desired_pos_df"]
# Filter to first risk dimension
for k, v in first_rd.items():
    pos_df = pos_df.filter(pl.col(k) == v)
pos_df = pos_df.drop_nulls(subset=["raw_desired_position"])
pos_ts = pos_df["timestamp"].to_list()

fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["raw_desired_position"].to_list(),
    name="Raw Desired Position",
    line=dict(width=1, color="rgba(99,110,250,0.4)"),
))
fig4.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["smoothed_desired_position"].to_list(),
    name="Smoothed Desired Position",
    line=dict(width=2),
))

fig4.update_layout(
    title=f"Desired Position — {rd_label}",
    template="plotly_dark",
    xaxis_title="Time",
    yaxis_title="Desired Position ($)",
)
fig4.show()